# Descriptive healthcare access analysis

This notebook reads the curated healthcare facility CSVs, the pipeline source/target parquet files, and the 150 km capped distance matrices for Portugal and the Netherlands. It computes descriptive access tables and example graphics, including:

- closest facility overall for each population point;
- closest layer 1 facility for each population point;
- closest layer 2 facility for each population point;
- closest layer 3 facility for each population point;
- per-hospital catchment summaries: percentage of national population for which each hospital is the closest layer 1 facility.

Layer convention used here:

- layer 1: hospital / hospital emergency;
- layer 2: huisartsenpost / permanent care / SAP-like urgent primary care;
- layer 3: general practitioner / primary care.

The distance matrices were produced with table-only sources, split outputs, no generated candidates, no extra amenity reads, and a 150 km cap.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

REPO = Path(r"C:\local\GIT\Public-Infrastructure-Service-Access")
SHARE_ROOT = Path(r"C:\Users\joaqu\OneDrive - UvA\share")

COUNTRIES = {
    "portugal": {
        "facility_csv": SHARE_ROOT / "Baoshan Liang" / "healthcare_access_analysis" / "facilities" / "portugal_health_service_locations_pipeline_with_permanent_care_geocoded.csv",
        "matrix": SHARE_ROOT / "Baoshan Liang" / "healthcare_access_analysis" / "distances" / "distance_matrix_src_table_dst_population_pop_1_sample_1_max_none_agg_20_maxdist_150000_amenity_amenity_all-dst_population-src_table_portugal_health_service_locations_pi_9220fa425f_no_candidates.parquet",
        "sources": SHARE_ROOT / "Baoshan Liang" / "healthcare_access_analysis" / "distances" / "sources_pop_1_sample_1_max_none_agg_20_maxdist_150000_amenity_amenity_all-dst_population-src_table_portugal_health_service_locations_pi_9220fa425f_no_candidates.parquet",
        "targets": SHARE_ROOT / "Baoshan Liang" / "healthcare_access_analysis" / "distances" / "targets_pop_1_sample_1_max_none_agg_20_maxdist_150000_amenity_amenity_all-dst_population-src_table_portugal_health_service_locations_pi_9220fa425f_no_candidates.parquet",
        "layer_map": {
            "hospital_emergency_official": "layer_1",
            "permanent_care_geocoded": "layer_2",
            "primary_care": "layer_3",
        },
    },
    "netherlands": {
        "facility_csv": SHARE_ROOT / "Jack Martin" / "healthcare_access_analysis" / "facilities" / "netherlands_health_service_locations_study_layers_european_nl.csv",
        "matrix": SHARE_ROOT / "Jack Martin" / "healthcare_access_analysis" / "distances" / "distance_matrix_src_table_dst_population_pop_1_sample_1_max_none_agg_20_maxdist_150000_amenity_amenity_all-dst_population-src_table_netherlands_health_service_locations_d979d68da5_no_candidates.parquet",
        "sources": SHARE_ROOT / "Jack Martin" / "healthcare_access_analysis" / "distances" / "sources_pop_1_sample_1_max_none_agg_20_maxdist_150000_amenity_amenity_all-dst_population-src_table_netherlands_health_service_locations_d979d68da5_no_candidates.parquet",
        "targets": SHARE_ROOT / "Jack Martin" / "healthcare_access_analysis" / "distances" / "targets_pop_1_sample_1_max_none_agg_20_maxdist_150000_amenity_amenity_all-dst_population-src_table_netherlands_health_service_locations_d979d68da5_no_candidates.parquet",
        "layer_map": {
            "hospital": "layer_1",
            "huisartsenpost": "layer_2",
            "general_practitioner": "layer_3",
        },
    },
}

OUT_DIR = SHARE_ROOT / "healthcare_access_descriptive_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for country, cfg in COUNTRIES.items():
    for key in ["facility_csv", "matrix", "sources", "targets"]:
        assert cfg[key].exists(), f"Missing {country} {key}: {cfg[key]}"

COUNTRIES

## Helper functions

In [ ]:
def weighted_quantile(values, weights, quantiles):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    if len(values) == 0:
        return [np.nan for _ in quantiles]
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumulative = np.cumsum(weights) - 0.5 * weights
    cumulative = cumulative / np.sum(weights)
    return np.interp(quantiles, cumulative, values).tolist()


def prepare_source_lookup(country):
    cfg = COUNTRIES[country]
    facilities = pd.read_csv(cfg["facility_csv"])
    facilities["facility_id"] = facilities["facility_id"].astype(str)
    facilities["source_id"] = "source_table_" + facilities["facility_id"]
    facilities["analysis_layer"] = facilities["layer"].map(cfg["layer_map"])
    facilities["facility_name"] = facilities["name"].fillna(facilities["facility_id"])

    sources = pd.read_parquet(cfg["sources"], columns=["ID", "Longitude", "Latitude", "dist_snap_source"])
    sources = sources.rename(columns={
        "ID": "source_id",
        "Longitude": "source_longitude",
        "Latitude": "source_latitude",
    })

    keep = [
        "source_id", "facility_id", "facility_name", "layer", "analysis_layer",
        "source", "source_detail", "amenity", "healthcare", "city", "postcode",
    ]
    keep = [col for col in keep if col in facilities.columns]
    lookup = sources.merge(facilities[keep], on="source_id", how="left")
    missing = lookup["analysis_layer"].isna().sum()
    if missing:
        print(f"Warning: {country} has {missing} source rows without an analysis layer.")
    return lookup


def prepare_targets(country):
    cfg = COUNTRIES[country]
    targets = pd.read_parquet(cfg["targets"], columns=["ID", "Longitude", "Latitude", "population"])
    return targets.rename(columns={
        "ID": "target_id",
        "Longitude": "target_longitude",
        "Latitude": "target_latitude",
    })


def _minimum_by_target(rows):
    if rows.empty:
        return rows
    return rows.loc[rows.groupby("target_id", sort=False)["total_dist"].idxmin()].copy()


def _update_best(existing, candidate):
    if candidate.empty:
        return existing
    candidate = _minimum_by_target(candidate)
    if existing is None or existing.empty:
        return candidate
    combined = pd.concat([existing, candidate], ignore_index=True)
    return _minimum_by_target(combined)


def compute_closest_tables(country, layers=("layer_1", "layer_2", "layer_3")):
    cfg = COUNTRIES[country]
    source_lookup = prepare_source_lookup(country)
    target_lookup = prepare_targets(country)
    layer_lookup = source_lookup[["source_id", "facility_id", "facility_name", "layer", "analysis_layer"]]

    best = {"overall": None, **{layer: None for layer in layers}}
    matrix_file = pq.ParquetFile(cfg["matrix"])
    columns = ["target_id", "source_id", "total_dist"]

    for group_idx in range(matrix_file.metadata.num_row_groups):
        chunk = matrix_file.read_row_group(group_idx, columns=columns).to_pandas()
        chunk = chunk.merge(layer_lookup, on="source_id", how="left")
        chunk = chunk.dropna(subset=["analysis_layer"])
        best["overall"] = _update_best(best["overall"], chunk)
        for layer in layers:
            best[layer] = _update_best(best[layer], chunk[chunk["analysis_layer"] == layer])

    closest = []
    for category, table in best.items():
        if table is None:
            continue
        table = table.merge(target_lookup, on="target_id", how="left")
        table["closest_category"] = category
        table["distance_km"] = table["total_dist"] / 1000.0
        closest.append(table)
    return pd.concat(closest, ignore_index=True), source_lookup, target_lookup


def access_summary(closest):
    rows = []
    for category, group in closest.groupby("closest_category", sort=False):
        q10, q25, q50, q75, q90 = weighted_quantile(
            group["distance_km"], group["population"], [0.10, 0.25, 0.50, 0.75, 0.90]
        )
        rows.append({
            "closest_category": category,
            "population_covered": group["population"].sum(),
            "population_points": group["target_id"].nunique(),
            "weighted_mean_km": np.average(group["distance_km"], weights=group["population"]),
            "weighted_p10_km": q10,
            "weighted_p25_km": q25,
            "weighted_median_km": q50,
            "weighted_p75_km": q75,
            "weighted_p90_km": q90,
        })
    return pd.DataFrame(rows)


def hospital_catchment_summary(closest):
    hospitals = closest[closest["closest_category"] == "layer_1"].copy()
    total_population = hospitals["population"].sum()
    summary = (
        hospitals.groupby(["source_id", "facility_id", "facility_name"], dropna=False)
        .agg(
            closest_population=("population", "sum"),
            population_points=("target_id", "nunique"),
            weighted_mean_distance_km=("distance_km", lambda s: np.average(s, weights=hospitals.loc[s.index, "population"])),
            min_distance_km=("distance_km", "min"),
            max_distance_km=("distance_km", "max"),
        )
        .reset_index()
    )
    summary["closest_population_pct"] = 100 * summary["closest_population"] / total_population
    return summary.sort_values("closest_population_pct", ascending=False)


def save_country_outputs(country, closest, source_lookup, target_lookup):
    country_dir = OUT_DIR / country
    country_dir.mkdir(parents=True, exist_ok=True)
    closest.to_parquet(country_dir / "closest_facility_by_population_point.parquet", index=False)
    source_lookup.to_csv(country_dir / "source_lookup_with_analysis_layers.csv", index=False)
    target_lookup.to_csv(country_dir / "target_population_points.csv", index=False)
    access_summary(closest).to_csv(country_dir / "distance_summary_by_closest_category.csv", index=False)
    hospital_catchment_summary(closest).to_csv(country_dir / "hospital_closest_population_share.csv", index=False)
    return country_dir

## Run the closest-facility analysis

This cell scans each distance matrix row group and keeps only the current minimum distance per population point and layer. On a normal workstation it is much lighter than materializing all matrix rows plus joins at once, but it still reads the full matrix files.

In [ ]:
results = {}

for country in COUNTRIES:
    print(f"Processing {country}...")
    closest, source_lookup, target_lookup = compute_closest_tables(country)
    output_dir = save_country_outputs(country, closest, source_lookup, target_lookup)
    results[country] = {
        "closest": closest,
        "source_lookup": source_lookup,
        "target_lookup": target_lookup,
        "access_summary": access_summary(closest),
        "hospital_catchments": hospital_catchment_summary(closest),
        "output_dir": output_dir,
    }
    print(f"  wrote outputs to {output_dir}")

## Descriptive tables

In [ ]:
for country, result in results.items():
    print(f"\n{country.upper()} distance summary")
    display(result["access_summary"])
    print(f"\n{country.upper()} hospitals serving the largest closest-population shares")
    display(result["hospital_catchments"].head(15))

## Graphics

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

for country, result in results.items():
    closest = result["closest"]
    fig, ax = plt.subplots(figsize=(9, 5))
    for category, group in closest.groupby("closest_category", sort=False):
        distances = group["distance_km"].clip(upper=150)
        ax.hist(distances, bins=50, alpha=0.45, weights=group["population"], label=category)
    ax.set_title(f"{country.title()}: population-weighted closest-facility distances")
    ax.set_xlabel("Distance to closest facility (km, capped at 150)")
    ax.set_ylabel("Population weight")
    ax.legend()
    fig.tight_layout()
    fig.savefig(result["output_dir"] / "weighted_distance_histogram.png", dpi=180)
    plt.show()

    top = result["hospital_catchments"].head(20).copy()
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top["facility_name"].iloc[::-1], top["closest_population_pct"].iloc[::-1])
    ax.set_title(f"{country.title()}: largest closest-hospital catchments")
    ax.set_xlabel("Share of national population for which hospital is closest (%)")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(result["output_dir"] / "top_hospital_closest_population_share.png", dpi=180)
    plt.show()

## Simple spatial context plots

These are not replacement cartographic context figures. They are quick checks showing target population points colored by nearest layer 1 distance and facility points by analysis layer.

In [ ]:
layer_colors = {"layer_1": "#d62728", "layer_2": "#1f77b4", "layer_3": "#2ca02c"}

for country, result in results.items():
    closest = result.get("closest")
    if closest is None or "closest_category" not in closest.columns:
        saved_closest = result.get("output_dir", OUT_DIR / country) / "closest_facility_by_population_point.parquet"
        if saved_closest.exists():
            closest = pd.read_parquet(saved_closest)
            result["closest"] = closest
        else:
            raise KeyError(
                f"{country}: result['closest'] does not contain closest_category. "
                "Run the closest-facility analysis cell before plotting."
            )

    layer1 = closest[closest["closest_category"] == "layer_1"].copy()
    if layer1.empty:
        print(f"Skipping {country}: no layer_1 closest rows available.")
        continue

    sources = result["source_lookup"].dropna(subset=["analysis_layer"])

    fig, ax = plt.subplots(figsize=(8, 8))
    sc = ax.scatter(
        layer1["target_longitude"],
        layer1["target_latitude"],
        c=layer1["distance_km"],
        s=np.clip(layer1["population"] / layer1["population"].quantile(0.95), 0.5, 12),
        cmap="viridis_r",
        alpha=0.65,
    )
    for layer, group in sources.groupby("analysis_layer"):
        ax.scatter(
            group["source_longitude"],
            group["source_latitude"],
            s=18 if layer == "layer_1" else 8,
            c=layer_colors.get(layer, "black"),
            label=layer,
            edgecolor="white",
            linewidth=0.25,
        )
    ax.set_title(f"{country.title()}: closest hospital distance and facility layers")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(loc="best")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Distance to closest layer 1 facility (km)")
    fig.tight_layout()
    fig.savefig(result["output_dir"] / "closest_hospital_distance_spatial_check.png", dpi=180)
    plt.show()